# Confocal Data Processing — Unified Workflow

**Complete pipeline for new confocal data batches:**

1. **Step 1 — Sort ND2 files by antibody pair** (Excel-based + hardcoded lookup)
2. **Step 2 — Separate 10X files** into `10X/` subfolders within each antibody folder
3. **Step 3 — Convert remaining 40X ND2 → TIFF** with max-Z projection, routed to `tiff_SC/` or `tiff_SNR/`

> **Note:** Step 3 replaces the ImageJ macro (`split_nd2_to_tiff_max_z.ijm`) with a pure-Python
> implementation using `nd2` + `tifffile`. No Fiji/ImageJ needed.

---

## Configuration

Edit these paths and settings once — all steps use them.

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

# ── PATHS ──────────────────────────────────────────────────────────────────────

EXCEL_FILE = r"C:\Users\Sapna Sinha\OneDrive - Massachusetts Institute of Technology\MIT\Surgery\Imaging\hand count\20260310_SampleAntibodyChart.xlsx"

INPUT_DIR  = r"E:\ND2\00_NEW\DHI_New_NEW"   # Folder containing all your new .nd2 files
OUTPUT_DIR = r"E:\ND2\00_NEW\DHI_New_NEW"   # Antibody subfolders will be created here

# ── BEHAVIOR ───────────────────────────────────────────────────────────────────

DRY_RUN    = True     # True = preview only; False = actually move/convert files
COPY_FILES = False    # True = copy instead of move (safer but uses more disk)

# ── ANTIBODY PAIRS ─────────────────────────────────────────────────────────────

ANTIBODY_PAIRS = [
    "CD3_CD4",
    "CD206_TRPV1",
    "MHCII_CD11c",
    "CD68_CD8",
    "GFAP_IBA1",
]

# Build antibody -> folder map (each individual antibody maps to its pair folder)
antibody_to_folder = {}
for pair in ANTIBODY_PAIRS:
    antibody_to_folder[pair] = pair          # full pair name maps to itself
    for ab in pair.split("_"):
        antibody_to_folder[ab] = pair        # individual antibody maps to pair

print("Antibody -> Folder mapping:")
for k, v in antibody_to_folder.items():
    print(f"  {k:15s} -> {v}")

input_path  = Path(INPUT_DIR)
output_path = Path(OUTPUT_DIR)

---
## Step 1A — Sort ND2 files by antibody pair (Excel-based)

Matches `.nd2` files to Excel rows using **(Animal ID, Section Location, Slide Number)**  
and moves them into the appropriate antibody-pair subfolder.

In [ ]:
# ── LOAD EXCEL ─────────────────────────────────────────────────────────────────

df = pd.read_excel(EXCEL_FILE, dtype=str)
df.columns = df.columns.str.strip()
print(f"Loaded {len(df)} rows.  Columns: {list(df.columns)}")

REQUIRED_COLS = ["Animal ID", "Section Location", "Slide Number", "Antibody"]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in Excel: {missing}\nFound: {list(df.columns)}")

# Build (AnimalID, Location, SlideNum) -> folder map from Excel
excel_slide_map = {}
unknown_abs = set()

for _, row in df.iterrows():
    animal   = str(row["Animal ID"]).strip()
    loc      = str(row["Section Location"]).strip()
    slide    = str(row["Slide Number"]).strip()
    antibody = str(row["Antibody"]).strip()

    if "nan" in (animal, loc, slide, antibody):
        continue

    folder = antibody_to_folder.get(antibody)
    if folder is None:
        unknown_abs.add(antibody)
        continue

    excel_slide_map[(animal, loc, slide)] = folder

print(f"Mapped {len(excel_slide_map)} unique slide keys from Excel.")
if unknown_abs:
    print(f"WARNING — Unknown antibodies (skipped): {sorted(unknown_abs)}")

In [ ]:
# ── SCAN & SORT via Excel ──────────────────────────────────────────────────────

nd2_files = [f for f in input_path.rglob("*.nd2") if f.is_file()]
print(f"Found {len(nd2_files)} .nd2 files in {INPUT_DIR}\n")

moved_p1     = 0
unmatched_p1 = []   # will be tried in Step 1B

for nd2 in nd2_files:
    parts = nd2.stem.split("_")
    if len(parts) < 4:
        unmatched_p1.append(nd2)
        continue

    # Try 4-part key first: (AnimalID, Location, SlideNum, SectionNum)
    key4 = tuple(parts[:4])
    # Also try 3-part key: (AnimalID, Location, SlideNum)
    key3 = tuple(parts[:3])

    folder_name = excel_slide_map.get(key4) or excel_slide_map.get(key3)

    if folder_name is None:
        unmatched_p1.append(nd2)
        continue

    dest_dir = output_path / folder_name
    dest     = dest_dir / nd2.name

    if DRY_RUN:
        print(f"  [DRY RUN P1] {nd2.name}  ->  {folder_name}/")
    else:
        dest_dir.mkdir(parents=True, exist_ok=True)
        if COPY_FILES:
            shutil.copy2(nd2, dest)
        else:
            shutil.move(str(nd2), dest)

    moved_p1 += 1

action = "Would move" if DRY_RUN else ("Copied" if COPY_FILES else "Moved")
print(f"\nPart 1 (Excel): {action} {moved_p1} files.")
print(f"  {len(unmatched_p1)} files not matched — will try hardcoded lookup next.")

---
## Step 1B — Sort remaining ND2 files (Hardcoded lookup)

For animals not covered by the Excel sheet.  
Key: `(AnimalID, Location, SlideNumber)` → folder name.  

**To add new animals:** just append entries to `SLIDE_LOOKUP` below.

In [ ]:
# ── HARDCODED SLIDE -> ANTIBODY LOOKUP ─────────────────────────────────────────
# Format: (AnimalID, Location, SlideNumber): "Antibody1_Antibody2"

SLIDE_LOOKUP = {
    # C65
    ("C65", "SC", "101"): "CD3_CD4",
    ("C65", "SC", "103"): "CD206_TRPV1",
    ("C65", "SC", "104"): "MHCII_CD11c",
    ("C65", "SC", "105"): "CD68_CD8",
    # C66
    ("C66", "SC", "101"): "CD3_CD4",
    ("C66", "SC", "102"): "CD206_TRPV1",
    ("C66", "SC", "103"): "MHCII_CD11c",
    ("C66", "SC", "105"): "CD68_CD8",
    ("C66", "SNR", "1"): "CD3_CD4",
    ("C66", "SNR", "2"): "CD206_TRPV1",
    ("C66", "SNR", "3"): "MHCII_CD11c",
    ("C66", "SNR", "5"): "CD68_CD8",
    # C67
    ("C67", "SC", "9"): "CD3_CD4",
    ("C67", "SC", "11"): "CD206_TRPV1",
    ("C67", "SC", "12"): "MHCII_CD11c",
    ("C67", "SC", "13"): "CD68_CD8",
    ("C67", "SC", "110"): "CD3_CD4",
    ("C67", "SC", "112"): "CD206_TRPV1",
    ("C67", "SC", "113"): "MHCII_CD11c",
    ("C67", "SC", "114"): "CD68_CD8",
    ("C67", "SNL", "6"): "CD3_CD4",
    ("C67", "SNL", "8"): "CD206_TRPV1",
    ("C67", "SNL", "9"): "MHCII_CD11c",
    ("C67", "SNL", "10"): "CD68_CD8",
    ("C67", "SNR", "6"): "CD3_CD4",
    ("C67", "SNR", "8"): "CD206_TRPV1",
    ("C67", "SNR", "9"): "MHCII_CD11c",
    ("C67", "SNR", "10"): "CD68_CD8",
    # C68
    ("C68", "SNR", "6"): "CD68_CD8",
    ("C68", "SNR", "7"): "CD68_CD8",
    ("C68", "SC", "106"): "CD68_CD8",
    ("C68", "SC", "101"): "CD3_CD4",
    ("C68", "SC", "102"): "CD206_TRPV1",
    ("C68", "SC", "103"): "MHCII_CD11c",
    ("C68", "SC", "104"): "CD68_CD8",
    ("C68", "SNR", "1"): "CD3_CD4",
    ("C68", "SNR", "2"): "CD206_TRPV1",
    ("C68", "SNR", "3"): "MHCII_CD11c",
    ("C68", "SNR", "4"): "CD68_CD8",
    ("C68", "SC", "107"): "GFAP_IBA1",
    # D77
    ("D77", "SC", "101"): "CD68_CD8",
    ("D77", "SNR", "1"): "CD68_CD8",
    ("D77", "SC", "102"): "GFAP_IBA1",
    ("D77", "SNR", "2"): "GFAP_IBA1",
    # D79
    ("D79", "SC", "101"): "CD3_CD4",
    ("D79", "SC", "102"): "CD206_TRPV1",
    ("D79", "SC", "103"): "MHCII_CD11c",
    ("D79", "SC", "104"): "CD68_CD8",
    ("D79", "SC", "105"): "GFAP_IBA1",
    ("D79", "SNR", "1"): "CD3_CD4",
    ("D79", "SNR", "2"): "CD206_TRPV1",
    ("D79", "SNR", "3"): "MHCII_CD11c",
    ("D79", "SNR", "4"): "CD68_CD8",
    ("D79", "SNR", "5"): "GFAP_IBA1",
    # D80
    ("D80", "SC", "101"): "CD3_CD4",
    ("D80", "SC", "102"): "CD206_TRPV1",
    ("D80", "SC", "103"): "MHCII_CD11c",
    ("D80", "SC", "104"): "CD68_CD8",
    ("D80", "SC", "105"): "GFAP_IBA1",
    ("D80", "SNR", "1"): "CD3_CD4",
    ("D80", "SNR", "2"): "CD206_TRPV1",
    ("D80", "SNR", "3"): "MHCII_CD11c",
    ("D80", "SNR", "4"): "CD68_CD8",
    ("D80", "SNR", "5"): "GFAP_IBA1",
    # H98
    ("H98", "SC", "101"): "CD3_CD4",
    ("H98", "SC", "102"): "CD206_TRPV1",
    ("H98", "SC", "103"): "MHCII_CD11c",
    ("H98", "SC", "104"): "CD68_CD8",
    ("H98", "SC", "105"): "GFAP_IBA1",
    ("H98", "SNR", "1"): "CD3_CD4",
    ("H98", "SNR", "2"): "CD206_TRPV1",
    ("H98", "SNR", "3"): "MHCII_CD11c",
    ("H98", "SNR", "4"): "CD68_CD8",
    ("H98", "SNR", "5"): "GFAP_IBA1",
    # H99
    ("H99", "SC", "101"): "CD68_CD8",
    ("H99", "SNR", "1"): "CD68_CD8",
    ("H99", "SC", "102"): "GFAP_IBA1",
    ("H99", "SNR", "2"): "GFAP_IBA1",
    ("H99", "SC", "104"): "CD3_CD4",
    ("H99", "SC", "105"): "CD206_TRPV1",
    ("H99", "SC", "106"): "MHCII_CD11c",
    ("H99", "SC", "107"): "CD68_CD8",
    ("H99", "SC", "108"): "GFAP_IBA1",
    ("H99", "SNR", "4"): "CD3_CD4",
    ("H99", "SNR", "5"): "CD206_TRPV1",
    ("H99", "SNR", "6"): "MHCII_CD11c",
    ("H99", "SNR", "7"): "CD68_CD8",
    ("H99", "SNR", "8"): "GFAP_IBA1",
    # I61
    ("I61", "SC", "109"): "CD68_CD8",
    ("I61", "SC", "110"): "CD68_CD8",
    ("I61", "SC", "111"): "CD68_CD8",
    ("I61", "SC", "6"): "MHCII_CD11c",
    ("I61", "SNR", "6"): "MHCII_CD11c",
    ("I61", "SC", "7"): "CD68_CD8",
    ("I61", "SNR", "7"): "CD68_CD8",
    ("I61", "SC", "112"): "CD68_CD8",
    ("I61", "SNR", "9"): "CD68_CD8",
    ("I61", "SC", "115"): "CD3_CD4",
    ("I61", "SC", "116"): "CD206_TRPV1",
    ("I61", "SC", "117"): "MHCII_CD11c",
    ("I61", "SC", "118"): "CD68_CD8",
    ("I61", "SC", "119"): "GFAP_IBA1",
    ("I61", "SNR", "15"): "CD3_CD4",
    ("I61", "SNR", "16"): "CD206_TRPV1",
    ("I61", "SNR", "17"): "MHCII_CD11c",
    ("I61", "SNR", "18"): "CD68_CD8",
    ("I61", "SNR", "19"): "GFAP_IBA1",
    ("I61", "SC", "114"): "GFAP_IBA1",
    # I85
    ("I85", "SC", "101"): "CD3_CD4",
    ("I85", "SC", "102"): "CD206_TRPV1",
    ("I85", "SC", "103"): "MHCII_CD11c",
    ("I85", "SC", "104"): "CD68_CD8",
    ("I85", "SC", "105"): "GFAP_IBA1",
    ("I85", "SNR", "1"): "CD3_CD4",
    ("I85", "SNR", "2"): "CD206_TRPV1",
    ("I85", "SNR", "3"): "MHCII_CD11c",
    ("I85", "SNR", "4"): "CD68_CD8",
    ("I85", "SNR", "5"): "GFAP_IBA1",
    # IM109
    ("IM109", "SC", "101"): "CD3_CD4",
    ("IM109", "SC", "103"): "CD206_TRPV1",
    ("IM109", "SC", "104"): "MHCII_CD11c",
    ("IM109", "SC", "105"): "CD68_CD8",
    ("IM109", "SNR", "1"): "CD3_CD4",
    ("IM109", "SNR", "3"): "CD206_TRPV1",
    ("IM109", "SNR", "4"): "MHCII_CD11c",
    ("IM109", "SNR", "5"): "CD68_CD8",
    ("IM109", "SC", "110"): "GFAP_IBA1",
    # IM118
    ("IM118", "SC", "6"): "CD3_CD4",
    ("IM118", "SC", "8"): "CD206_TRPV1",
    ("IM118", "SC", "9"): "MHCII_CD11c",
    ("IM118", "SC", "10"): "CD68_CD8",
    ("IM118", "SC", "106"): "CD3_CD4",
    ("IM118", "SC", "108"): "CD206_TRPV1",
    ("IM118", "SC", "109"): "MHCII_CD11c",
    ("IM118", "SC", "110"): "CD68_CD8",
    ("IM118", "SNL", "6"): "CD3_CD4",
    ("IM118", "SNL", "8"): "CD206_TRPV1",
    ("IM118", "SNL", "9"): "MHCII_CD11c",
    ("IM118", "SNL", "10"): "CD68_CD8",
    ("IM118", "SNR", "6"): "CD3_CD4",
    ("IM118", "SNR", "8"): "CD206_TRPV1",
    ("IM118", "SNR", "9"): "MHCII_CD11c",
    ("IM118", "SNR", "10"): "CD68_CD8",
    ("IM118", "SC", "116"): "MHCII_CD11c",
    ("IM118", "SC", "117"): "CD68_CD8",
    ("IM118", "SNR", "17"): "CD68_CD8",
    ("IM118", "SC", "111"): "GFAP_IBA1",
    # IM119
    ("IM119", "SC", "101"): "CD3_CD4",
    ("IM119", "SC", "102"): "CD206_TRPV1",
    ("IM119", "SC", "103"): "MHCII_CD11c",
    ("IM119", "SC", "104"): "CD68_CD8",
    ("IM119", "SNR", "7"): "CD3_CD4",
    ("IM119", "SNR", "8"): "CD206_TRPV1",
    ("IM119", "SNR", "9"): "MHCII_CD11c",
    ("IM119", "SNR", "10"): "CD68_CD8",
    # IM121
    ("IM121", "SC", "101"): "CD3_CD4",
    ("IM121", "SC", "105"): "CD206_TRPV1",
    ("IM121", "SC", "107"): "MHCII_CD11c",
    ("IM121", "SC", "109"): "CD68_CD8",
    ("IM121", "SNR", "1"): "CD3_CD4",
    ("IM121", "SNR", "5"): "CD206_TRPV1",
    ("IM121", "SNR", "7"): "MHCII_CD11c",
    ("IM121", "SNR", "9"): "CD68_CD8",
    ("IM121", "SC", "110"): "CD68_CD8",
    # IMC136
    ("IMC136", "SC", "101"): "CD68_CD8",
    ("IMC136", "SNR", "1"): "CD68_CD8",
    ("IMC136", "TAR", "51"): "CD68_CD8",
    ("IMC136", "GR", "1"): "CD68_CD8",
    ("IMC136", "SC", "102"): "GFAP_IBA1",
    ("IMC136", "SNR", "2"): "GFAP_IBA1",
    ("IMC136", "TAR", "52"): "GFAP_IBA1",
    ("IMC136", "GR", "2"): "GFAP_IBA1",
    # IMCP138
    ("IMCP138", "SC", "101"): "CD68_CD8",
    ("IMCP138", "SNR", "1"): "CD68_CD8",
    ("IMCP138", "TAR", "51"): "CD68_CD8",
    ("IMCP138", "GR", "1"): "CD68_CD8",
    ("IMCP138", "SC", "102"): "GFAP_IBA1",
    ("IMCP138", "SNR", "2"): "GFAP_IBA1",
    ("IMCP138", "TAR", "52"): "GFAP_IBA1",
    ("IMCP138", "GR", "2"): "GFAP_IBA1",
    # P46
    ("P46", "SC", "101"): "CD3_CD4",
    ("P46", "SC", "103"): "CD206_TRPV1",
    ("P46", "SC", "104"): "MHCII_CD11c",
    ("P46", "SC", "105"): "CD68_CD8",
    ("P46", "SNR", "1"): "CD3_CD4",
    ("P46", "SNR", "3"): "CD206_TRPV1",
    ("P46", "SNR", "4"): "MHCII_CD11c",
    ("P46", "SNR", "5"): "CD68_CD8",
    ("P46", "SC", "6"): "CD3_CD4",
    ("P46", "SC", "8"): "CD206_TRPV1",
    ("P46", "SC", "9"): "MHCII_CD11c",
    ("P46", "SC", "10"): "CD68_CD8",
    ("P46", "SC", "106"): "CD68_CD8",
    ("P46", "SC", "114"): "CD68_CD8",
    ("P46", "SNR", "6"): "CD68_CD8",
    ("P46", "SC", "107"): "CD3_CD4",
    ("P46", "SC", "108"): "CD206_TRPV1",
    ("P46", "SC", "109"): "MHCII_CD11c",
    ("P46", "SC", "111"): "CD68_CD8",
    ("P46", "SC", "112"): "GFAP_IBA1",
    # P47
    ("P47", "SNL", "2"): "CD3_CD4",
    ("P47", "Spleen", "41"): "CD3_CD4",
    ("P47", "Spleen", "43"): "CD206_TRPV1",
    ("P47", "Spleen", "44"): "MHCII_CD11c",
    ("P47", "Spleen", "45"): "CD68_CD8",
    ("P47", "Spleen", "50"): "GFAP_IBA1",
    # P50
    ("P50", "SC", "7"): "CD3_CD4",
    ("P50", "SC", "8"): "CD206_TRPV1",
    ("P50", "SC", "9"): "MHCII_CD11c",
    ("P50", "SC", "10"): "CD68_CD8",
    ("P50", "SNL", "7"): "CD3_CD4",
    ("P50", "SNL", "8"): "CD206_TRPV1",
    ("P50", "SNL", "9"): "MHCII_CD11c",
    ("P50", "SNL", "10"): "CD68_CD8",
    ("P50", "SC", "11"): "MHCII_CD11c",
    ("P50", "SNR", "11"): "MHCII_CD11c",
    ("P50", "SC", "12"): "CD68_CD8",
    ("P50", "SNR", "12"): "CD68_CD8",
    ("P50", "SC", "13"): "GFP_GFP",
    ("P50", "SC", "14"): "GFP_GFP",
    ("P50", "SNR", "13"): "CD68_CD8",
    ("P50", "SC", "101"): "CD3_CD4",
    ("P50", "SC", "105"): "CD206_TRPV1",
    ("P50", "SC", "107"): "MHCII_CD11c",
    ("P50", "SC", "109"): "CD68_CD8",
    ("P50", "SC", "116"): "GFAP_IBA1",
    ("P50", "SNR", "1"): "CD3_CD4",
    ("P50", "SNR", "5"): "CD206_TRPV1",
    ("P50", "SNR", "7"): "MHCII_CD11c",
    ("P50", "SNR", "9"): "CD68_CD8",
    ("P50", "SNR", "16"): "GFAP_IBA1",
    ("P50", "SC", "2"): "GFAP_IBA1",
    # WT
    ("WT", "SNR", "6"): "MHCII_CD11c",
    ("WT", "SNR", "1"): "CD3_CD4",
    ("WT", "SNR", "3"): "CD206_TRPV1",
    ("WT", "SNR", "4"): "MHCII_CD11c",
    ("WT", "SNR", "5"): "CD68_CD8",
    # WT2
    ("WT2", "SC", "101"): "CD3_CD4",
    ("WT2", "SC", "103"): "CD206_TRPV1",
    ("WT2", "SC", "104"): "MHCII_CD11c",
    ("WT2", "SC", "105"): "CD68_CD8",
    ("WT2", "SC", "106"): "CD68_CD8",
    ("WT2", "SC", "111"): "MHCII_CD11c",
    ("WT2", "SC", "112"): "CD68_CD8",
    ("WT2", "SNR", "7"): "CD68_CD8",
    ("WT2", "SC", "109"): "GFAP_IBA1",
    # WT3
    ("WT3", "SC", "106"): "CD68_CD8",
    ("WT3", "SNR", "6"): "CD68_CD8",
    ("WT3", "SC", "112"): "GFAP_IBA1",
    ("WT3", "SNR", "7"): "GFAP_IBA1",
    ("WT3", "SC", "101"): "CD3_CD4",
    ("WT3", "SC", "102"): "CD206_TRPV1",
    ("WT3", "SC", "103"): "MHCII_CD11c",
    ("WT3", "SC", "104"): "CD68_CD8",
    ("WT3", "SNR", "1"): "CD3_CD4",
    ("WT3", "SNR", "2"): "CD206_TRPV1",
    ("WT3", "SNR", "3"): "MHCII_CD11c",
    ("WT3", "SNR", "4"): "CD68_CD8",
    # WT4
    ("WT4", "SC", "101"): "CD3_CD4",
    ("WT4", "SC", "102"): "CD206_TRPV1",
    ("WT4", "SC", "103"): "MHCII_CD11c",
    ("WT4", "SC", "104"): "CD68_CD8",
    ("WT4", "SC", "105"): "GFAP_IBA1",
    ("WT4", "SNR", "1"): "CD3_CD4",
    ("WT4", "SNR", "2"): "CD206_TRPV1",
    ("WT4", "SNR", "3"): "MHCII_CD11c",
    ("WT4", "SNR", "4"): "CD68_CD8",
    ("WT4", "SNR", "5"): "GFAP_IBA1",
}

print(f"Hardcoded lookup contains {len(SLIDE_LOOKUP)} slide entries.")

In [ ]:
# ── MATCH REMAINING FILES via hardcoded lookup ─────────────────────────────────

moved_p2     = 0
still_no_key = []

for nd2 in unmatched_p1:
    parts = nd2.stem.split("_")
    if len(parts) < 3:
        still_no_key.append(nd2.name)
        continue

    key = (parts[0], parts[1], parts[2])  # (AnimalID, Location, SlideNum)
    folder_name = SLIDE_LOOKUP.get(key)

    if folder_name is None:
        still_no_key.append(nd2.name)
        continue

    dest_dir = output_path / folder_name
    dest     = dest_dir / nd2.name

    if DRY_RUN:
        print(f"  [DRY RUN P2] {nd2.name}  ->  {folder_name}/")
    else:
        dest_dir.mkdir(parents=True, exist_ok=True)
        if COPY_FILES:
            shutil.copy2(nd2, dest)
        else:
            shutil.move(str(nd2), dest)

    moved_p2 += 1

# ── STEP 1 SUMMARY ─────────────────────────────────────────────────────────────
action = "Would move" if DRY_RUN else ("Copied" if COPY_FILES else "Moved")
print(f"\n{'='*55}")
print(f"  STEP 1 SUMMARY")
print(f"  Part A (Excel):     {action} {moved_p1} files")
print(f"  Part B (Hardcoded): {action} {moved_p2} files")
print(f"  Total sorted:       {moved_p1 + moved_p2} files")

if still_no_key:
    print(f"\n  Unmatched (left in place): {len(still_no_key)}")
    for f in sorted(still_no_key)[:15]:
        print(f"      - {f}")
    if len(still_no_key) > 15:
        print(f"      ... and {len(still_no_key) - 15} more")

print(f"{'='*55}")

---
## Step 2 — Move 10X files into `10X/` subfolders

Inside each antibody folder, any file with `10X` in its name gets moved into a `10X/` subfolder.

In [ ]:
# ── SEPARATE 10X FILES ─────────────────────────────────────────────────────────

parent = Path(OUTPUT_DIR)
subfolders = [f for f in parent.iterdir() if f.is_dir()]
print(f"Found {len(subfolders)} subfolders in {OUTPUT_DIR}\n")

total_10x = 0

for subfolder in sorted(subfolders):
    # Only look at top-level files (not already in 10X/ subfolder)
    files_10x = [f for f in subfolder.iterdir()
                 if f.is_file() and "10X" in f.name]

    if not files_10x:
        print(f"  {subfolder.name}: no 10X files, skipping.")
        continue

    dest_dir = subfolder / "10X"
    print(f"  {subfolder.name}: {len(files_10x)} files -> 10X/")

    for f in sorted(files_10x):
        dest = dest_dir / f.name
        if DRY_RUN:
            print(f"    [DRY RUN] {f.name}")
        else:
            dest_dir.mkdir(parents=True, exist_ok=True)
            if COPY_FILES:
                shutil.copy2(f, dest)
            else:
                shutil.move(str(f), dest)

    total_10x += len(files_10x)

action = "Would move" if DRY_RUN else ("Copied" if COPY_FILES else "Moved")
print(f"\n{'='*50}")
print(f"  Step 2: {action} {total_10x} 10X files total.")
print(f"{'='*50}")

---
## Step 3 — Convert remaining 40X ND2 → TIFF (max-Z projection)

This replaces the ImageJ macro `split_nd2_to_tiff_max_z.ijm`.  
For each antibody subfolder, remaining `.nd2` files (the 40X ones) are:
1. Opened with the `nd2` library
2. Max-intensity Z-projected per channel, per series/location
3. Saved as `.tif` into `tiff_SC/` or `tiff_SNR/` based on the filename

**Requires:** `pip install nd2 tifffile numpy`

In [ ]:
# Uncomment to install if needed:
# !pip install nd2 tifffile numpy

In [ ]:
import numpy as np
import nd2
import tifffile

def nd2_to_maxz_tiff(nd2_path, output_dir, dry_run=False):
    """
    Open an ND2 file, compute max-Z projection for each series/position,
    and save as a multi-channel TIFF (matching the ImageJ macro behavior).

    Output filename: {stem}_loc{NN}.tif
    """
    stem = nd2_path.stem
    output_dir = Path(output_dir)
    saved = 0

    with nd2.ND2File(str(nd2_path)) as f:
        # Check if multi-position (series)
        sizes = f.sizes  # dict like {'P': 3, 'Z': 20, 'C': 4, 'Y': 1024, 'X': 1024}
        n_positions = sizes.get('P', 1)

        if n_positions > 1:
            # Multi-position: iterate over each
            for p in range(n_positions):
                out_name = f"{stem}_loc{p+1:02d}.tif"
                out_path = output_dir / out_name

                if dry_run:
                    print(f"    [DRY RUN] -> {out_name}")
                    saved += 1
                    continue

                # Read this position's data
                # f.read_frame() or slice the dask array
                data = np.asarray(f.asarray())

                # data shape depends on axes; find Z axis and max-project
                axes = list(f.sizes.keys())
                if 'P' in axes and 'Z' in axes:
                    p_idx = axes.index('P')
                    z_idx = axes.index('Z')
                    # Select position p, then max over Z
                    slices = [slice(None)] * len(axes)
                    slices[p_idx] = p
                    pos_data = data[tuple(slices)]
                    # Now Z axis shifted by 1 if P was before Z
                    new_z_idx = z_idx - 1 if p_idx < z_idx else z_idx
                    max_proj = pos_data.max(axis=new_z_idx)
                elif 'P' in axes:
                    p_idx = axes.index('P')
                    slices = [slice(None)] * len(axes)
                    slices[p_idx] = p
                    max_proj = data[tuple(slices)]
                else:
                    max_proj = data

                output_dir.mkdir(parents=True, exist_ok=True)
                tifffile.imwrite(str(out_path), max_proj)
                saved += 1
        else:
            # Single position
            out_name = f"{stem}_loc01.tif"
            out_path = output_dir / out_name

            if dry_run:
                print(f"    [DRY RUN] -> {out_name}")
                return 1

            data = np.asarray(f.asarray())
            axes = list(f.sizes.keys())
            if 'Z' in axes:
                z_idx = axes.index('Z')
                max_proj = data.max(axis=z_idx)
            else:
                max_proj = data

            output_dir.mkdir(parents=True, exist_ok=True)
            tifffile.imwrite(str(out_path), max_proj)
            saved = 1

    return saved

print("nd2_to_maxz_tiff() defined.")

In [ ]:
# ── CONVERT REMAINING ND2 -> TIFF (per antibody subfolder) ─────────────────────

parent = Path(OUTPUT_DIR)
subfolders = [f for f in parent.iterdir() if f.is_dir()]

total_converted = 0
total_skipped   = 0

for subfolder in sorted(subfolders):
    # Only process top-level .nd2 files (10X already moved away in Step 2)
    nd2_files = [f for f in subfolder.iterdir()
                 if f.is_file() and f.suffix.lower() == ".nd2"]

    if not nd2_files:
        continue

    print(f"\n{subfolder.name}: {len(nd2_files)} ND2 files to convert")

    for nd2_file in sorted(nd2_files):
        name = nd2_file.name

        # Route to tiff_SNR/ or tiff_SC/ based on filename
        if "SNR" in name or "SNL" in name:
            out_dir = subfolder / "tiff_SNR"
        elif "SC" in name:
            out_dir = subfolder / "tiff_SC"
        else:
            # Other locations (Spleen, TAR, GR, etc.) — put in tiff_other/
            out_dir = subfolder / "tiff_other"

        try:
            n = nd2_to_maxz_tiff(nd2_file, out_dir, dry_run=DRY_RUN)
            total_converted += n
            if not DRY_RUN:
                print(f"  {name} -> {out_dir.name}/ ({n} TIFF(s))")
        except Exception as e:
            print(f"  ERROR processing {name}: {e}")
            total_skipped += 1

print(f"\n{'='*55}")
action = "Would create" if DRY_RUN else "Created"
print(f"  Step 3: {action} {total_converted} TIFF files from ND2.")
if total_skipped:
    print(f"  Errors: {total_skipped} files failed (see above).")
print(f"{'='*55}")

---
## Final Summary

In [ ]:
print("\n" + "#"*60)
print("  WORKFLOW COMPLETE")
print("#"*60)
print(f"\n  Step 1A (Excel sort):     {moved_p1} files")
print(f"  Step 1B (Hardcoded sort):  {moved_p2} files")
print(f"  Step 2  (10X separation):  {total_10x} files")
print(f"  Step 3  (ND2 -> TIFF):     {total_converted} TIFFs created")
if DRY_RUN:
    print(f"\n  >>> DRY RUN — no files were moved or created. <<<")
    print(f"  >>> Set DRY_RUN = False in the Configuration cell and re-run. <<<")
else:
    print(f"\n  All done!")
print("\n" + "#"*60)